# 模块大作业：连锁门店补货预警矩阵

## 任务来函

区域仓库每天面对数十个“门店 × 商品”组合，却只能优先处理少量补货任务。你需要建立一套 NumPy 预警矩阵，快速识别库存覆盖不足的位置，形成优先清单，并抽样检查预警规则是否过于激进。

> 这是一份需要由你继续完成的项目 Notebook。系统只提供任务、里程碑和少量代码起点；请自行新增 Markdown 与代码单元，保留关键输出，并解释你的选择。

## 你已经拥有的项目零件

| 已学阶段 | 章节 | 可带入作业的项目零件 |
| --- | --- | --- |
| 数组建模 | 第 16–17 章 | ndarray、shape、dtype 与轴含义 |
| 定位片段 | 第 18–19 章 | 切片、索引、布尔掩码与形状调整 |
| 批量规则 | 第 20 章 | 向量化和广播计算 |
| 复核决策 | 第 21 章 | 按轴统计、随机抽样与可复现性 |

模块作业不是重新开始。请从前面章节选择可复用的规则、数据结构、分析表、图表草稿或验证方法，并在新增的 Markdown 单元中写明“复用了什么、做了什么调整”。


## 本章目标

| 完成后能够 | 对应完成证据 |
|---|---|
| 把门店与商品映射到二维数组的轴，解释 shape、dtype、标签和库存单位。 | 里程碑 1：库存与需求矩阵、轴标签；手工定位一个门店和商品，核对数值。 |
| 用广播、掩码和索引计算库存覆盖或缺口，并把预警位置还原为业务标签。 | 里程碑 2：结果矩阵、风险掩码和预警清单；用一个非方阵手算核对广播方向。 |
| 根据风险排序并用可复现抽样复核规则，说明阈值改变如何影响补货行动。 | 里程碑 3：按轴统计、优先清单、随机种子及一次阈值前后对照。 |


## 学习准备与补学路径

先独立尝试下面的小任务。遇到困难时回看对应章节，再返回当前里程碑；它们不另设章节作业，也不单独计分。

| 遇到的问题 | 回看章节 | 再做一次 |
|---|---|---|
| 能算出数却说不清对应门店或商品 | [第17章 数组基础（ndarray）](/course/chapter-17) | 创建 2×3 矩阵，分别打印两个轴的标签，指出第 2 家店第 3 件商品的位置。 |
| 广播成功但业务方向错误 | [第20章 向量化与广播](/course/chapter-20) | 将长度为 3 的商品阈值应用于 2×3 库存矩阵，逐列手算一行进行对照。 |
| 每次抽查结果不同，无法复核 | [第21章 统计计算与随机抽样](/course/chapter-21) | 使用固定种子的生成器独立抽样两次，打印样本索引并解释 seed 的作用。 |


## 任务合同：数据、边界与交付

**数据与边界：** 可以根据任务自建一组规模适中的库存、近期需求和安全库存数组，也可以从课程零售数据整理后再转为 ndarray。必须保存门店和商品标签，并说明所有数组的轴顺序、单位和广播方向。

**必须完成：**

1. 建立至少两个 shape 兼容的二维业务矩阵，并保留轴标签。
2. 使用索引、切片或布尔掩码定位具体门店/商品组合。
3. 通过广播计算库存差额、覆盖天数或同等级风险指标。
4. 使用按轴统计和排序形成补货优先级。
5. 使用固定随机种子抽样复核一部分未预警记录。

**最终交付：**

- 带轴说明和关键 shape 检查的 Notebook。
- 补货预警矩阵、优先补货清单和门店/商品统计摘要。
- 抽样复核记录，以及阈值可能造成的误报/漏报说明。


## 如何开始：三级提示

### 第一层｜操作路线

1. 先打印数组的 `shape`、`dtype` 和少量样本。
2. 用布尔掩码定位缺失/异常位置。
3. 用向量化聚合得到每一列或每一路的指标。

### 第二层｜代码起点

下面只给出 API 或结构起点，字段、参数、规则、异常处理和结果解释均由你完成。

```python
import numpy as np

values = np.asarray(...)  # TODO：说明两个轴的含义
missing_mask = np.isnan(values)
# TODO：定义异常掩码，并按轴计算统计量
```

### 第三层｜遇到问题时检查

先检查输入数据与中间结果，再检查字段、shape、粒度、排序或指标口径。不要通过删除校验条件来让结果“看起来正确”。


## 里程碑 1｜建立库存与需求矩阵

**承接章节：** 第 16–17 章：ndarray、shape 与 dtype

**此刻的项目情境：** 区域仓库每天要判断哪些门店和商品存在缺货风险。你需要先把库存、近期开单量和安全库存整理成含义稳定的数组。

**你要解决的问题：** 数组的每个轴和每个数值代表什么，标签怎样与矩阵位置保持一致？

**完成证据：** 库存矩阵、需求矩阵、门店/商品标签、shape/dtype 检查和样本切片。

**容易失分的地方：** 只保留数值矩阵却丢失行列标签，或让两个矩阵的轴顺序不一致。

完成代码后，请自行新增一个 Markdown 单元，按“观察到什么 → 这说明什么 → 下一步怎么做”解释结果。


<!-- math-foundation:capstone-numpy -->
### 数学推导｜补货缺口与库存覆盖天数

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜估计提前期需求。** 日需求为 $d_{ij}$、提前期为 $\ell_j$ 时，等待补货期间预计消耗

$$
D^{lead}_{ij}=d_{ij}\ell_j
$$

**第 2 步｜加上安全缓冲。** 目标库存为 $R_{ij}=D^{lead}_{ij}+s_j$。

**第 3 步｜从两个方向判断风险。** 库存缺口 $G_{ij}=I_{ij}-R_{ij}$ 衡量“够不够”，覆盖天数 $C_{ij}=I_{ij}/d_{ij}$ 衡量“还能撑多久”。

**把上面的关系收束为本章计算式：**

$$
R_{ij}=s_j+d_{ij}\ell_j,\qquad G_{ij}=I_{ij}-R_{ij},\qquad C_{ij}=\frac{I_{ij}}{d_{ij}}
$$

**符号解释：** $I$ 是库存，$d$ 是日需求，$s$ 是安全库存，$\ell$ 是提前期；$G<0$ 表示预警。

**代码对应：** 利用广播计算 `required_stock`、`stock_gap` 和 `coverage_days`。

**使用边界：** 需求为 0 时覆盖天数需要安全除法；参数变化应做敏感性检查。


In [ ]:
# 里程碑 1｜建立库存与需求矩阵
# 这段代码应解决：数组的每个轴和每个数值代表什么，标签怎样与矩阵位置保持一致？
# 完成后应留下：库存矩阵、需求矩阵、门店/商品标签、shape/dtype 检查和样本切片。
# TODO：从下面的起点继续；关键规则和设计选择需要写注释。

import numpy as np

stock = np.asarray(...)
demand = np.asarray(...)
store_names = np.asarray(...)
product_names = np.asarray(...)
# TODO：验证两个矩阵的 shape，并说明 axis=0 / axis=1


## 里程碑 2｜批量计算缺货风险

**承接章节：** 第 18–20 章：索引、形状、向量化与广播

**此刻的项目情境：** 不同商品拥有不同安全库存和补货提前期。运营人员希望一次看到所有“门店 × 商品”的库存覆盖天数和预警位置。

**你要解决的问题：** 怎样用广播、布尔掩码和索引完成批量风险计算，并定位到具体门店和商品？

**完成证据：** 覆盖天数或库存差额矩阵；风险掩码；高风险位置及标签；一次有业务意义的转置或形状调整。

**容易失分的地方：** 广播虽然能够运行，但安全库存对应错了轴；使用双层循环逐格计算。

完成代码后，请自行新增一个 Markdown 单元，按“观察到什么 → 这说明什么 → 下一步怎么做”解释结果。


In [ ]:
# 里程碑 2｜批量计算缺货风险
# 这段代码应解决：怎样用广播、布尔掩码和索引完成批量风险计算，并定位到具体门店和商品？
# 完成后应留下：覆盖天数或库存差额矩阵；风险掩码；高风险位置及标签；一次有业务意义的转置或形状调整。
# TODO：从下面的起点继续；关键规则和设计选择需要写注释。

safety_stock = np.asarray(...)  # TODO：说明它对应商品轴还是门店轴
# TODO：利用广播计算库存差额或覆盖天数
alert_mask = ...
# TODO：用 np.where / 布尔索引定位高风险组合


## 里程碑 3｜形成补货清单并抽样复核

**承接章节：** 第 21 章：统计计算与随机抽样

**此刻的项目情境：** 仓库只能优先处理有限数量的预警。你需要依据风险强度排序，并随机抽取部分普通记录检查规则是否过度预警。

**你要解决的问题：** 哪些按轴统计、排序和抽样结果能够支持一份可解释的补货清单？

**完成证据：** 门店/商品统计摘要；优先补货组合；固定随机种子的复核样本；阈值局限说明。

**容易失分的地方：** 只报告总体平均值；抽样没有固定随机种子；补货顺序无法回溯到风险指标。

完成代码后，请自行新增一个 Markdown 单元，按“观察到什么 → 这说明什么 → 下一步怎么做”解释结果。


In [ ]:
# 里程碑 3｜形成补货清单并抽样复核
# 这段代码应解决：哪些按轴统计、排序和抽样结果能够支持一份可解释的补货清单？
# 完成后应留下：门店/商品统计摘要；优先补货组合；固定随机种子的复核样本；阈值局限说明。
# TODO：从下面的起点继续；关键规则和设计选择需要写注释。

rng = np.random.default_rng(2026)
# TODO：计算门店或商品维度的统计量
# TODO：按风险指标生成优先清单
# TODO：随机抽取普通记录复核，并解释抽样范围


## 交付、挑战与自查

**基础提交清单：**

- [ ] 所有矩阵都能对应回具体门店和商品。
- [ ] 核心预警计算使用向量化或广播完成。
- [ ] 广播前后的 shape 与业务轴已经验证。
- [ ] 抽样使用固定随机种子并说明抽样范围。

### 进阶挑战（可选）

比较两组安全库存或补货提前期参数，说明预警数量如何变化，以及更保守的规则会增加什么成本。

挑战任务必须建立在基础任务已经完整、可复现的前提上；不能用额外图表或复杂模型掩盖基础证据缺失。


## 评分标准（100 分）

### 共同能力：30 分

- **可复现性（10 分）**：重启内核后能按顺序运行，路径和依赖清楚。
- **证据与注释（10 分）**：关键代码说明设计原因，结论能回到具体输出。
- **边界与诚实表达（10 分）**：说明数据来源、假设、限制，不把相关性写成确定因果。

### 本模块核心能力：70 分

- **数组建模与轴解释（20 分）**：shape、dtype、标签和每个轴的业务含义一致。
- **索引、掩码与形状操作（15 分）**：能够定位具体业务片段，并解释形状变化。
- **广播与向量化（20 分）**：批量计算对应正确业务轴，避免不必要的逐格循环。
- **统计、抽样与验证（15 分）**：统计口径合理，抽样可复现，预警结果能回到具体位置。

### 提交门槛

Notebook 必须能够运行；错误或异常记录不能被静默隐藏；关键结论必须可以回溯。最后的确认单元只检查摘要是否填写，不替代教师评分。

<!-- module-teaching-assessment -->
### 达标与返工规则

- 总分至少 60/100，共同能力至少 18/30，模块核心能力至少 42/70，且下列关键门槛全部满足，才视为达标。
- **本模块关键门槛：** 预警必须对应正确的门店和商品；对零需求、缺失或非法数值明确处理策略，不能把无穷值当作正常覆盖天数。
- 每项按证据给分：独立完成且处理边界为该项满分；主流程正确但证据不全为约 75%；最小流程可复现为约 60%；未完成或关键方法错误为 0–50%。教师须记录扣分依据。
- 学生作品须在不运行参考答案的情况下，重启内核并从头复现；参考答案中产生的变量、文件和输出不能作为自己的完成证据。
- 运行进度、摘要填写和勾选状态仅是学习记录，不是自动评分。关键门槛未满足时先返工再评，不用其他高分抵消。
- **可选迁移：** 将一个商品的补货提前期增加一天，比较缺口与排序；说明预警不等于确定会缺货。 迁移不另设加分，不挤占必做任务；可作为相应维度的边界或解释证据。


In [ ]:
# 提交前确认：请在完成三个里程碑后填写。
# 教师将结合代码、输出、解释和导出文件评分，不会只看本单元。
submission_summary = {
    "任务与使用者": "",
    "三个里程碑的完成证据": "",
    "最重要的结果或作品功能": "",
    "限制与下一步": "",
}

missing = [
    key
    for key, value in submission_summary.items()
    if not str(value).strip() or str(value).strip() == "..."
]
if missing:
    raise ValueError("请先填写提交摘要：" + "、".join(missing))
print("提交摘要已填写；请重启内核并从头运行，再对照评分标准检查证据。")


In [ ]:
# 参考答案｜里程碑 1：建立库存与需求矩阵
# 默认隐藏。建议先完成自己的实现，再展开对照设计选择。
# 三个答案 Cell 前后衔接；阅读时请按里程碑顺序理解变量与中间结果。

# 参考答案：连锁门店补货预警矩阵
from pathlib import Path
import numpy as np

OUTPUT_DIR = Path("output/numpy_replenishment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(2026)

store_names = np.array(["东城店", "西城店", "南站店", "北湖店"])
product_names = np.array(["咖啡", "牛奶", "面包", "纸巾", "洗衣液", "矿泉水"])

# axis=0 对应门店，axis=1 对应商品。
stock = rng.integers(
    8, 90, size=(len(store_names), len(product_names))
).astype(float)
daily_demand = rng.integers(2, 18, size=stock.shape).astype(float)
safety_stock = np.array([18, 22, 20, 15, 12, 30], dtype=float)  # 对应商品轴
lead_days = np.array([3, 2, 2, 4, 5, 2], dtype=float)  # 对应商品轴


In [ ]:
# 参考答案｜里程碑 2：批量计算缺货风险
# 默认隐藏。建议先完成自己的实现，再展开对照设计选择。
# 三个答案 Cell 前后衔接；阅读时请按里程碑顺序理解变量与中间结果。

# 广播：每个门店都使用同一组商品安全库存与补货提前期。
required_stock = safety_stock + daily_demand * lead_days
stock_gap = stock - required_stock
coverage_days = np.divide(
    stock,
    daily_demand,
    out=np.full_like(stock, np.inf),
    where=daily_demand > 0,
)
alert_mask = stock_gap < 0

store_pos, product_pos = np.where(alert_mask)
risk_score = np.where(
    alert_mask, -stock_gap / np.maximum(required_stock, 1), 0
)


In [ ]:
# 参考答案｜里程碑 3：形成补货清单并抽样复核
# 默认隐藏。建议先完成自己的实现，再展开对照设计选择。
# 三个答案 Cell 前后衔接；阅读时请按里程碑顺序理解变量与中间结果。

priority_order = np.argsort(risk_score[store_pos, product_pos])[::-1]
priority_rows = []
for index in priority_order:
    i, j = store_pos[index], product_pos[index]
    priority_rows.append(
        [
            store_names[i],
            product_names[j],
            stock[i, j],
            daily_demand[i, j],
            required_stock[i, j],
            stock_gap[i, j],
            risk_score[i, j],
        ]
    )

# axis=1 聚合每家门店；axis=0 聚合每种商品。
store_alert_count = alert_mask.sum(axis=1)
product_alert_rate = alert_mask.mean(axis=0)

# 从未触发预警的组合中固定抽样，检查规则是否遗漏明显风险。
normal_positions = np.argwhere(~alert_mask)
sample_size = min(6, len(normal_positions))
sample_positions = normal_positions[
    rng.choice(len(normal_positions), size=sample_size, replace=False)
]

priority_path = OUTPUT_DIR / "replenishment_priority.csv"
header = "store,product,stock,daily_demand,required_stock,stock_gap,risk_score"
if priority_rows:
    np.savetxt(
        priority_path,
        np.asarray(priority_rows, dtype=object),
        delimiter=",",
        fmt="%s",
        header=header,
        comments="",
        encoding="utf-8",
    )
else:
    priority_path.write_text(header + "\n", encoding="utf-8")

print("stock shape / dtype:", stock.shape, stock.dtype)
print("各门店预警数:", dict(zip(store_names, store_alert_count)))
print("各商品预警率:", dict(zip(product_names, product_alert_rate.round(3))))
print("优先补货前 5 项:", priority_rows[:5])
print("普通组合复核位置:", sample_positions.tolist())
print("导出:", priority_path)
